# Đánh giá tác động của 2 cơ chế NLI trong kiểm tra tuân thủ GRI

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns

EXP_DIR = Path.cwd()
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))
import nli_lib as lib

for d in (lib.OUT_DIR, lib.TABLES_DIR, lib.STATS_DIR, lib.FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

REQUIRED = [
    lib.PAIRING_CSV,
    lib.adjudicated_csv("v2"),
    lib.HUMAN_REVIEW_DONE_CSV,
    lib.HUMAN_REVIEW_V2_CSV,
    lib.SAMPLE_MAIN_CSV,
    lib.PRE_FIX_PAIRING_CSV,
    lib.LATENCY_OBSERVED_CSV,
]
missing = [str(f) for f in REQUIRED if not f.exists()]
if missing:
    raise FileNotFoundError(f"Missing inputs: {missing}")
print("[OK] All required inputs present")


[OK] All required inputs present


## Post-fix các variant trên 150 case human


In [2]:
df_15 = lib.load_postfix_sample(view="15")
df_14 = lib.load_postfix_sample(view="14")
# print(f"15-report view: n={len(df_15)} cases")
# print(f"14-report view: n={len(df_14)} cases (Energean2024 excluded)")
# print("\nPhase bucket distribution (14-report):")
# print(df_14["bucket"].value_counts().to_string())


In [3]:
acc_14 = lib.build_accuracy_table_postfix(df_14, label="14-report")
acc_15 = lib.build_accuracy_table_postfix(df_15, label="15-report")
acc_tbl = pd.concat([acc_14, acc_15], ignore_index=True)
print("Bảng tab:c4-postfix-acc — Per-variant accuracy (14-report, phase=ALL):")
print(
    acc_14[acc_14["phase"] == "ALL"][["variant", "n", "accuracy", "ci_low", "ci_high", "kappa"]]
    .to_string(index=False)
)
print("\nPer-variant accuracy by phase bucket (14-report):")
print(
    acc_14[acc_14["phase"] != "ALL"][["variant", "phase", "n", "accuracy", "kappa"]]
    .to_string(index=False)
)
acc_tbl.to_csv(lib.TABLES_DIR / "T9_postfix_accuracy.csv", index=False)
print(f"\nSaved: {lib.TABLES_DIR / 'T9_postfix_accuracy.csv'}")


Bảng tab:c4-postfix-acc — Per-variant accuracy (14-report, phase=ALL):
variant   n  accuracy  ci_low  ci_high  kappa
     a0 137     0.577   0.489    0.664  0.378
     a1 137     0.635   0.555    0.715  0.456
     a2 137     0.474   0.394    0.555  0.246
  v_new 137     0.467   0.380    0.547  0.238

Per-variant accuracy by phase bucket (14-report):
variant      phase  n  accuracy  kappa
     a0     phase3 29     0.586  0.376
     a0 phase5_3-3 65     0.677  0.530
     a0     phase6 42     0.405  0.099
     a1     phase3 29     0.690  0.500
     a1 phase5_3-3 65     0.692  0.545
     a1     phase6 42     0.500  0.238
     a2     phase3 29     0.414  0.200
     a2 phase5_3-3 65     0.523  0.326
     a2     phase6 42     0.452  0.144
  v_new     phase3 29     0.345  0.117
  v_new phase5_3-3 65     0.508  0.300
  v_new     phase6 42     0.500  0.219

Saved: d:\Final_GRAG\experiments\module3_nli\outputs\tables\T9_postfix_accuracy.csv


In [4]:
mcn = lib.build_mcnemar_table_postfix(df_14, label="14-report")
print("Bảng tab:c4-tn1-mcnemar — Pairwise McNemar (post-fix, 14-report, ALL + phase5_3-3):")
print(
    mcn[mcn["scope"].isin(["ALL", "phase5_3-3"])]
    [["pair", "scope", "n", "b", "c", "p_value", "significant"]]
    .to_string(index=False)
)
mcn.to_csv(lib.TABLES_DIR / "T11_postfix_mcnemar.csv", index=False)
print(f"\nSaved: {lib.TABLES_DIR / 'T11_postfix_mcnemar.csv'}")


Bảng tab:c4-tn1-mcnemar — Pairwise McNemar (post-fix, 14-report, ALL + phase5_3-3):
       pair      scope   n  b  c  p_value  significant
   a0 vs a1        ALL 137 13 21   0.2295        False
   a0 vs a1 phase5_3-3  65  5  6   1.0000        False
   a0 vs a2        ALL 137 38 24   0.0980        False
   a0 vs a2 phase5_3-3  65 15  5   0.0414         True
a0 vs v_new        ALL 137 38 23   0.0722        False
a0 vs v_new phase5_3-3  65 15  4   0.0192         True
   a1 vs a2        ALL 137 47 25   0.0128         True
   a1 vs a2 phase5_3-3  65 18  7   0.0433         True
a1 vs v_new        ALL 137 46 23   0.0076         True
a1 vs v_new phase5_3-3  65 18  6   0.0227         True
a2 vs v_new        ALL 137 18 17   1.0000        False
a2 vs v_new phase5_3-3  65 10  9   1.0000        False

Saved: d:\Final_GRAG\experiments\module3_nli\outputs\tables\T11_postfix_mcnemar.csv


In [5]:
n_all = len(df_14)
n_p5  = int((df_14["bucket"] == "phase5_3-3").sum())
mde_all = lib.power_mde_paired(n=n_all, p_disc=0.30, alpha=0.05, target_power=0.80)
mde_p5  = lib.power_mde_paired(n=n_p5,  p_disc=0.30, alpha=0.05, target_power=0.80)
print(f"Power MDE (paired, α=0.05, power=0.80, p_disc=0.30):")
print(f"  ALL (n={n_all}):        MDE = {mde_all['mde_pp']} pp  (thesis L254)")
print(f"  Phase 5 3-3 (n={n_p5}): MDE = {mde_p5['mde_pp']} pp")
(lib.STATS_DIR / "power_mde.json").write_text(
    json.dumps({"all": mde_all, "phase5_3-3": mde_p5}, indent=2), encoding="utf-8"
)
print(f"Saved: {lib.STATS_DIR / 'power_mde.json'}")


Power MDE (paired, α=0.05, power=0.80, p_disc=0.30):
  ALL (n=137):        MDE = 13.0 pp  (thesis L254)
  Phase 5 3-3 (n=65): MDE = 18.69 pp
Saved: d:\Final_GRAG\experiments\module3_nli\outputs\stats\power_mde.json


In [6]:
all_14 = acc_14[acc_14["phase"] == "ALL"].set_index("variant")["accuracy"]
all_15 = acc_15[acc_15["phase"] == "ALL"].set_index("variant")["accuracy"]
sens = pd.DataFrame({
    "acc_14": all_14,
    "acc_15": all_15,
    "delta_pp": (all_14 - all_15) * 100,
})
print("Sensitivity 14-report vs 15-report (accuracy ALL, thesis L231):")
print(sens.round(4).to_string())
rank_14 = list(all_14.sort_values(ascending=False).index)
rank_15 = list(all_15.sort_values(ascending=False).index)
print(f"\nRanking 14-report: {rank_14}")
print(f"Ranking 15-report: {rank_15}")
sens.to_csv(lib.TABLES_DIR / "T12_sensitivity_14_vs_15.csv")
print(f"Saved: {lib.TABLES_DIR / 'T12_sensitivity_14_vs_15.csv'}")


Sensitivity 14-report vs 15-report (accuracy ALL, thesis L231):
         acc_14  acc_15  delta_pp
variant                          
a0        0.577   0.560       1.7
a1        0.635   0.600       3.5
a2        0.474   0.473       0.1
v_new     0.467   0.467       0.0

Ranking 14-report: ['a1', 'a0', 'a2', 'v_new']
Ranking 15-report: ['a1', 'a0', 'a2', 'v_new']
Saved: d:\Final_GRAG\experiments\module3_nli\outputs\tables\T12_sensitivity_14_vs_15.csv


In [7]:
VARIANTS = lib.VARIANTS
VLABEL   = lib.VARIANT_LABELS
COLORS   = {"a0": "#4C72B0", "a1": "#55A868", "a2": "#C44E52", "v_new": "#8172B2"}
H = "human_correct_status"

phases = ["phase3", "phase5_3-3", "phase6"]
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(phases))
width = 0.18
for i, v in enumerate(VARIANTS):
    accs, errs_lo, errs_hi = [], [], []
    for b in phases:
        sub = df_14[df_14["bucket"] == b]
        correct = (sub[f"{v}_post"] == sub[H]).astype(int).values
        acc, lo, hi = lib._bootstrap_ci_simple(correct)
        accs.append(acc * 100); errs_lo.append((acc - lo) * 100); errs_hi.append((hi - acc) * 100)
    ax.bar(x + i * width, accs, width, label=VLABEL[v], color=COLORS[v], yerr=[errs_lo, errs_hi], capsize=3, alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([p.replace("_", " ") for p in phases])
ax.set_ylabel("Accuracy (%)")
ax.set_title("F6 — Variant accuracy by phase (post bug-fix, 14-report, 95% CI)")
ax.legend(loc="upper left", fontsize=9); ax.set_ylim(0, 100); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(lib.FIGURES_DIR / "F6_accuracy_by_phase_14r.png", dpi=140)
plt.close()

phases_ = ["ALL", "phase3", "phase5_3-3", "phase6"]
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(phases_))
for i, v in enumerate(VARIANTS):
    ks = []
    for b in phases_:
        sub = df_14 if b == "ALL" else df_14[df_14["bucket"] == b]
        ks.append(lib.cohen_kappa(sub[f"{v}_post"], sub[H], labels=lib.STATUS_LABELS))
    ax.bar(x + i * width, ks, width, label=VLABEL[v], color=COLORS[v], alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([p.replace("_", " ") for p in phases_])
ax.set_ylabel("Cohen's kappa vs human")
ax.set_title("F7 — Kappa post bug-fix (14-report)")
ax.axhline(0.4, color="green", linestyle="--", alpha=0.5, label="moderate (0.40)")
ax.axhline(0.6, color="darkgreen", linestyle="--", alpha=0.5, label="substantial (0.60)")
ax.legend(loc="upper left", fontsize=8); ax.set_ylim(-0.1, 0.8); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(lib.FIGURES_DIR / "F7_kappa_panel_14r.png", dpi=140)
plt.close()
print(f"Saved: F6_accuracy_by_phase_14r.png, F7_kappa_panel_14r.png")


Saved: F6_accuracy_by_phase_14r.png, F7_kappa_panel_14r.png


## Bug-fix lift pre/post


In [8]:
lift_14 = lib.build_lift_table_postfix(df_14, label="14-report")
lift_15 = lib.build_lift_table_postfix(df_15, label="15-report")
lift_tbl = pd.concat([lift_14, lift_15], ignore_index=True)
print("Bảng tab:c4-tn1-lift — Bug-fix paired lift (14-report, ALL + Phase 5 3-3):")
print(
    lift_14[lift_14["phase"].isin(["ALL", "phase5_3-3"])]
    [["variant", "phase", "n", "pre_acc", "post_acc", "lift_pp", "mcnemar_p"]]
    .to_string(index=False)
)
lift_tbl.to_csv(lib.TABLES_DIR / "T10_postfix_lift.csv", index=False)
print(f"\nSaved: {lib.TABLES_DIR / 'T10_postfix_lift.csv'}")

panel = lib.kappa_pre_post_panel(df_14)
print("\nBảng tab:c4-tn1-kappa-lift — κ pre vs post bug-fix (14-report):")
print(panel[panel["phase"].isin(["ALL", "phase5_3-3"])].to_string(index=False))
panel.to_csv(lib.STATS_DIR / "kappa_pre_post_panel.csv", index=False)
print(f"\nSaved: {lib.STATS_DIR / 'kappa_pre_post_panel.csv'}")


Bảng tab:c4-tn1-lift — Bug-fix paired lift (14-report, ALL + Phase 5 3-3):
variant      phase   n  pre_acc  post_acc  lift_pp  mcnemar_p
     a0        ALL 137    0.518     0.577     5.84     0.2153
     a0 phase5_3-3  65    0.554     0.677    12.31     0.1849
     a1        ALL 137    0.504     0.635    13.14     0.0014
     a1 phase5_3-3  65    0.431     0.692    26.15     0.0015
     a2        ALL 137    0.409     0.474     6.57     0.0784
     a2 phase5_3-3  65    0.415     0.523    10.77     0.1671
  v_new        ALL 137    0.416     0.467     5.11     0.2295
  v_new phase5_3-3  65    0.415     0.508     9.23     0.2863

Saved: d:\Final_GRAG\experiments\module3_nli\outputs\tables\T10_postfix_lift.csv

Bảng tab:c4-tn1-kappa-lift — κ pre vs post bug-fix (14-report):
variant      phase   n  kappa_pre  kappa_post  delta_kappa
     a0        ALL 137     0.2720      0.3779       0.1059
     a0 phase5_3-3  65     0.2980      0.5301       0.2322
     a1        ALL 137     0.2543      0.45

In [9]:
rows_f = []
for v in VARIANTS:
    for b in ["ALL", "phase5_3-3"]:
        sub = df_14 if b == "ALL" else df_14[df_14["bucket"] == b]
        c_pre  = (sub[f"{v}_pre"]  == sub[H]).astype(float).values
        c_post = (sub[f"{v}_post"] == sub[H]).astype(float).values
        lift = (c_post.mean() - c_pre.mean()) * 100
        res = scipy.stats.bootstrap(
            (c_pre, c_post),
            lambda a, b: (b.mean() - a.mean()) * 100,
            n_resamples=2000, method="percentile", paired=True, random_state=42,
        )
        rows_f.append({
            "label": f"{VLABEL[v]} ({b})", "lift": lift,
            "lo": float(res.confidence_interval.low), "hi": float(res.confidence_interval.high), "variant": v,
        })
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(rows_f))
for i, r in enumerate(rows_f):
    ax.errorbar(r["lift"], i, xerr=[[r["lift"] - r["lo"]], [r["hi"] - r["lift"]]],
                fmt="o", color=COLORS[r["variant"]], capsize=4, markersize=8)
ax.set_yticks(y); ax.set_yticklabels([r["label"] for r in rows_f])
ax.set_xlabel("Bug-fix lift (post − pre, percentage points)")
ax.axvline(0, color="black", alpha=0.4)
ax.set_title("F8 — Bug-fix lift (14-report, 95% bootstrap CI)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(lib.FIGURES_DIR / "F8_lift_forest_14r.png", dpi=140)
plt.close()
print(f"Saved: {lib.FIGURES_DIR / 'F8_lift_forest_14r.png'}")


Saved: d:\Final_GRAG\experiments\module3_nli\outputs\figures\F8_lift_forest_14r.png


## §4 HINT mechanism + cost


In [10]:
hint = lib.hint_over_prediction_summary(df_14)
print("Bảng tab:c4-overprediction-ratio — HINT over-prediction no_evidence (14-report):")
print(hint.to_string(index=False))
(lib.STATS_DIR / "hint_over_prediction.json").write_text(
    hint.to_json(orient="records", indent=2), encoding="utf-8"
)
print(f"Saved: {lib.STATS_DIR / 'hint_over_prediction.json'}")

lat = lib.latency_per_variant()
print("\nBảng tab:c4-overprediction-cost — Latency per variant (14 reports):")
print(lat.to_string(index=False))
lat.to_csv(lib.TABLES_DIR / "T13_latency_per_variant.csv", index=False)
print(f"Saved: {lib.TABLES_DIR / 'T13_latency_per_variant.csv'}")


Bảng tab:c4-overprediction-ratio — HINT over-prediction no_evidence (14-report):
variant  n_actual_no_evidence  n_predicted_no_evidence  ratio
     a0                    36                       53  1.472
     a1                    36                       44  1.222
     a2                    36                       81  2.250
  v_new                    36                       83  2.306
Saved: d:\Final_GRAG\experiments\module3_nli\outputs\stats\hint_over_prediction.json

Bảng tab:c4-overprediction-cost — Latency per variant (14 reports):
variant  n  median_minutes  mean_minutes  stdev_minutes  total_minutes
     a0 14           38.94         38.09           7.28         533.27
     a1 14           38.25         37.46           6.39         524.50
     a2 14           43.16         40.56           7.20         567.78
  v_new 14           42.93         42.66          11.19         597.29
Saved: d:\Final_GRAG\experiments\module3_nli\outputs\tables\T13_latency_per_variant.csv


In [11]:
labels_cm = ["no_evidence", "partial", "pass"]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, v in zip(axes, VARIANTS):
    ct = lib.confusion_matrix(df_14[f"{v}_post"], df_14[H], labels_cm)
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False, linewidths=0.5,
                xticklabels=labels_cm, yticklabels=labels_cm)
    ax.set_xlabel("human"); ax.set_ylabel("predicted" if v == "a0" else "")
    ax.set_title(VLABEL[v])
    ax.tick_params(axis="x", rotation=30); ax.tick_params(axis="y", rotation=0)
plt.suptitle("F9 — Confusion matrices (post bug-fix, 14-report)")
plt.tight_layout()
plt.savefig(lib.FIGURES_DIR / "F9_confusion_heatmap_14r.png", dpi=140)
plt.close()
print(f"Saved: {lib.FIGURES_DIR / 'F9_confusion_heatmap_14r.png'}")


Saved: d:\Final_GRAG\experiments\module3_nli\outputs\figures\F9_confusion_heatmap_14r.png
